# 📊 Semana 5 — Experimentos A/B, Resultados y Validación
## Tesis: Asistente Conversacional Inteligente para iTimeControl

**Entregables cubiertos en este notebook:**
- ✅ Experimentos ejecutados (A/B): Baseline vs Var1 vs Var2
- ✅ Resultados comparables con tabla estándar y gráfico Recall@k
- ✅ Feature set y pipeline (features añadidas/quitadas, fit solo en train)
- ✅ Confirmación de cero leakage y split correcto
- ✅ Validación: Cross Validation 5-fold + Holdout

---
### Diseño experimental
| Experimento | Descripción | Cambio respecto al anterior |
|---|---|---|
| **Baseline** | TF-IDF unigrams | — |
| **Var1** | TF-IDF bigrams + normalización | +bigrams, +strip_accents, +sublinear_tf |
| **Var2** | TF-IDF bigrams + reranking | +reranking por longitud, +filtro similitud |


---
## 1. Setup e importaciones

In [ ]:
import sys, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt', quiet=True)
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = ['#4C72B0', '#55A868', '#C44E52', '#DD8452']
random.seed(42)
np.random.seed(42)

ROOT     = Path('..')
LOGS_DIR = ROOT / 'logs'
LOGS_DIR.mkdir(exist_ok=True)
print('✅ Setup completado')

---
## 2. Carga del corpus y split sin leakage

In [ ]:
import json
from pathlib import Path

# Cargar dataset real de iTimeControl
datasets_dir = ROOT / 'data' / 'datasets'

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

train_data = load_jsonl(datasets_dir / 'train.jsonl')
val_data   = load_jsonl(datasets_dir / 'val.jsonl')
test_data  = load_jsonl(datasets_dir / 'test.jsonl')

X_train = [r['instruction'] for r in train_data]
X_val   = [r['instruction'] for r in val_data]
X_test  = [r['instruction'] for r in test_data]

# Corpus de respuestas (documentos a recuperar) — solo de TRAIN
corpus_train = [r['output'] for r in train_data]
refs_test    = [r['output'] for r in test_data]

print('='*50)
print('  SPLIT DEL DATASET — iTimeControl')
print('='*50)
print(f'  Train    : {len(X_train):4d} ejemplos  (80%)')
print(f'  Val      : {len(X_val):4d} ejemplos  (10%)')
print(f'  Test     : {len(X_test):4d} ejemplos  (10%)')
print(f'  Total    : {len(X_train)+len(X_val)+len(X_test):4d} ejemplos')
print('='*50)
print(f'\n✅ CERO LEAKAGE:')
print(f'   TF-IDF se ajusta (fit) SOLO sobre train ({len(X_train)} muestras)')
print(f'   Evaluación sobre test ({len(X_test)} muestras) — nunca visto por el modelo')
print(f'   Split tipo: holdout aleatorio estratificado')

# Verificación de leakage
train_set = set(X_train)
test_set  = set(X_test)
overlap   = train_set & test_set
print(f'\n  Verificación overlap train-test: {len(overlap)} coincidencias exactas')
print(f'  → {"⚠️ LEAKAGE DETECTADO" if overlap else "✅ Sin leakage"}')

---
## 3. Feature set y pipeline — qué se añade en cada variante

In [ ]:
# Tabla de features por experimento
feature_table = pd.DataFrame({
    'Feature / Transformación': [
        'ngram_range (unigrams)',
        'ngram_range (bigrams)',
        'strip_accents=unicode',
        'sublinear_tf (log TF)',
        'max_features=8000',
        'max_features=10000',
        'Reranking por longitud doc',
        'Filtro similitud mínima',
        'Fit solo en TRAIN',
    ],
    'Baseline': ['✅','❌','❌','❌','❌','❌','❌','❌','✅'],
    'Var1':     ['✅','✅','✅','✅','✅','❌','❌','❌','✅'],
    'Var2':     ['✅','✅','✅','✅','❌','✅','✅','✅','✅'],
})
feature_table = feature_table.set_index('Feature / Transformación')
print('FEATURE SET POR EXPERIMENTO:')
print(feature_table.to_string())
print('\nNota: fit solo en train → confirma cero leakage en los 3 experimentos')

---
## 4. Experimentos A/B — Entrenamiento y evaluación

In [ ]:
# Funciones de métricas
def get_rouge(pred, ref):
    s = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
    r = s.score(ref, pred)
    return r['rouge1'].fmeasure, r['rouge2'].fmeasure, r['rougeL'].fmeasure

def get_bleu(pred, ref):
    p, r = pred.lower().split(), ref.lower().split()
    return sentence_bleu([r], p, smoothing_function=SmoothingFunction().method1) if p and r else 0.0

def recall_at_k(results, ref, k):
    ref_t = set(ref.lower().split())
    for doc, _ in results[:k]:
        if len(ref_t & set(doc.lower().split())) / len(ref_t) > 0.25:
            return 1.0
    return 0.0

def evaluate(retriever_fn, queries, refs, k_list=[1,3,5]):
    r1s,r2s,rLs,bleus = [],[],[],[]
    rk = {k:[] for k in k_list}
    times = []
    for q, ref in zip(queries, refs):
        t0 = time.time()
        res = retriever_fn(q, max(k_list))
        times.append(time.time()-t0)
        pred = res[0][0] if res else ''
        r1,r2,rL = get_rouge(pred, ref)
        r1s.append(r1); r2s.append(r2); rLs.append(rL)
        bleus.append(get_bleu(pred, ref))
        for k in k_list:
            rk[k].append(recall_at_k(res, ref, k))
    return {
        'ROUGE-1': round(np.mean(r1s),4), 'ROUGE-2': round(np.mean(r2s),4),
        'ROUGE-L': round(np.mean(rLs),4), 'BLEU': round(np.mean(bleus),4),
        **{f'Recall@{k}': round(np.mean(rk[k]),4) for k in k_list},
        'Latencia_ms': round(np.mean(times)*1000,2),
    }

# ── BASELINE ─────────────────────────────────────────────────────────────────
print('Entrenando Baseline...')
tfidf_base = TfidfVectorizer(ngram_range=(1,1), max_features=5000)
M_base = tfidf_base.fit_transform(corpus_train)  # fit SOLO en train
def ret_base(q,k):
    v = tfidf_base.transform([q])
    s = cosine_similarity(v,M_base).flatten()
    top = s.argsort()[::-1][:k]
    return [(corpus_train[i],float(s[i])) for i in top]
m_base = evaluate(ret_base, X_test, refs_test)
print(f'  ✅ Baseline   ROUGE-1={m_base["ROUGE-1"]} Recall@3={m_base["Recall@3"]}')

# ── VARIANTE 1 ───────────────────────────────────────────────────────────────
print('Entrenando Variante 1 (bigrams + normalización)...')
tfidf_v1 = TfidfVectorizer(ngram_range=(1,2), max_features=8000,
                            strip_accents='unicode', sublinear_tf=True)
M_v1 = tfidf_v1.fit_transform(corpus_train)  # fit SOLO en train
def ret_v1(q,k):
    v = tfidf_v1.transform([q])
    s = cosine_similarity(v,M_v1).flatten()
    top = s.argsort()[::-1][:k]
    return [(corpus_train[i],float(s[i])) for i in top]
m_v1 = evaluate(ret_v1, X_test, refs_test)
print(f'  ✅ Variante 1 ROUGE-1={m_v1["ROUGE-1"]} Recall@3={m_v1["Recall@3"]}')

# ── VARIANTE 2 ───────────────────────────────────────────────────────────────
print('Entrenando Variante 2 (bigrams + reranking)...')
tfidf_v2 = TfidfVectorizer(ngram_range=(1,2), max_features=10000,
                            strip_accents='unicode', sublinear_tf=True)
M_v2 = tfidf_v2.fit_transform(corpus_train)  # fit SOLO en train
doc_lens = np.array([len(d.split()) for d in corpus_train])
len_bonus = np.clip(doc_lens / doc_lens.max(), 0.8, 1.0)
def ret_v2(q,k):
    v = tfidf_v2.transform([q])
    s = cosine_similarity(v,M_v2).flatten()
    adj = s * len_bonus
    top = adj.argsort()[::-1][:k]
    return [(corpus_train[i],float(adj[i])) for i in top if adj[i]>=0.05]
m_v2 = evaluate(ret_v2, X_test, refs_test)
print(f'  ✅ Variante 2 ROUGE-1={m_v2["ROUGE-1"]} Recall@3={m_v2["Recall@3"]}')

print('\n✅ Todos los experimentos completados')

---
## 5. Tabla estándar de resultados

In [ ]:
metrics_show = ['ROUGE-1','ROUGE-2','ROUGE-L','BLEU','Recall@1','Recall@3','Recall@5','Latencia_ms']

df_results = pd.DataFrame({
    'Métrica':    metrics_show,
    'Baseline':   [m_base[m] for m in metrics_show],
    'Var1 (+bigrams+norm)': [m_v1[m] for m in metrics_show],
    'Var2 (+reranking)':    [m_v2[m] for m in metrics_show],
})

def highlight_best(row):
    vals = row[['Baseline','Var1 (+bigrams+norm)','Var2 (+reranking)']]
    if row['Métrica'] == 'Latencia_ms':
        best = vals.idxmin()
    else:
        best = vals.idxmax()
    return ['' if col not in vals.index else ('font-weight: bold; color: green' if col==best else '') for col in row.index]

print('TABLA ESTÁNDAR — Baseline / Var1 / Var2')
print('(negrita = mejor valor en cada métrica)')
print(df_results.to_string(index=False))

# Guardar en CSV para el log
df_results.to_csv('../logs/tabla_resultados_semana5.csv', index=False)
print('\n✅ Tabla guardada en logs/tabla_resultados_semana5.csv')

---
## 6. Gráfico clave — Recall@k vs k

In [ ]:
k_vals = [1, 3, 5]
r_base = [m_base[f'Recall@{k}'] for k in k_vals]
r_v1   = [m_v1[f'Recall@{k}']   for k in k_vals]
r_v2   = [m_v2[f'Recall@{k}']   for k in k_vals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_vals, r_base, 'o-', color=COLORS[0], lw=2.5, ms=9, label='Baseline (TF-IDF unigrams)')
ax.plot(k_vals, r_v1,   's-', color=COLORS[1], lw=2.5, ms=9, label='Var1: +bigrams +normalización')
ax.plot(k_vals, r_v2,   '^-', color=COLORS[2], lw=2.5, ms=9, label='Var2: +reranking +filtro')

for vals, color in zip([r_base, r_v1, r_v2], COLORS):
    for k, v in zip(k_vals, vals):
        ax.annotate(f'{v:.2f}', (k,v), textcoords='offset points',
                    xytext=(0,12), ha='center', fontsize=10, color=color, fontweight='bold')

ax.set_xlabel('k (documentos recuperados)', fontsize=12)
ax.set_ylabel('Recall@k', fontsize=12)
ax.set_title('Recall@k vs k — Experimentos A/B iTimeControl Assistant', fontsize=13)
ax.set_xticks(k_vals)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../logs/recall_at_k_semana5.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfico guardado: logs/recall_at_k_semana5.png')

---
## 7. Validación — Cross Validation 5-fold + Holdout

In [ ]:
# Asignar intenciones al corpus completo
INTENT_KW = {
    'registro_asistencia': ['registrar','marcar','asistencia','entrada','salida','marcacion'],
    'reportes':            ['reporte','informe','exportar','excel','estadistica','descargar'],
    'horarios':            ['horario','turno','jornada','calendario','tolerancia'],
    'empleados':           ['empleado','personal','nuevo','agregar'],
    'solicitudes':         ['permiso','vacacion','ausencia','justificar','solicitud'],
    'configuracion':       ['configurar','backup','rol','dispositivo','feriado','contrasena'],
}

all_questions = [r['instruction'] for r in train_data + val_data + test_data]

def assign_intent(text):
    tl = text.lower()
    best, sc = 'general', 0
    for intent, kws in INTENT_KW.items():
        s = sum(1 for kw in kws if kw in tl)
        if s > sc: best, sc = intent, s
    return best

all_labels = [assign_intent(q) for q in all_questions]

from collections import Counter
print('Distribución de intenciones:')
for intent, cnt in sorted(Counter(all_labels).items(), key=lambda x: -x[1]):
    bar = '█' * cnt
    print(f'  {intent:<25} {cnt:3d}  {bar}')

# Cross-Validation estratificada 5-fold
nb_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, strip_accents='unicode')),
    ('clf',   MultinomialNB(alpha=0.5))
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(nb_pipe, all_questions, all_labels, cv=skf)

print('\nCROSS-VALIDATION 5-fold — Naive Bayes (clasificación de intención):')
print(classification_report(all_labels, y_pred_cv, zero_division=0))

cv_metrics = {
    'accuracy':  round(float(np.mean([y_pred_cv[i]==all_labels[i] for i in range(len(all_labels))])), 4),
    'precision': round(float(precision_score(all_labels, y_pred_cv, average='weighted', zero_division=0)), 4),
    'recall':    round(float(recall_score(all_labels, y_pred_cv, average='weighted', zero_division=0)), 4),
    'f1':        round(float(f1_score(all_labels, y_pred_cv, average='weighted', zero_division=0)), 4),
}
print(f'Accuracy: {cv_metrics["accuracy"]} | F1: {cv_metrics["f1"]} | Precision: {cv_metrics["precision"]}')

In [ ]:
# Curva de precisión por fold (visual de CV)
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(all_questions, all_labels), 1):
    X_tr = [all_questions[i] for i in tr_idx]
    X_vl = [all_questions[i] for i in val_idx]
    y_tr = [all_labels[i] for i in tr_idx]
    y_vl = [all_labels[i] for i in val_idx]
    nb_pipe.fit(X_tr, y_tr)
    acc = np.mean(np.array(nb_pipe.predict(X_vl)) == np.array(y_vl))
    fold_scores.append(acc)
    print(f'  Fold {fold}: accuracy={acc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Validación — Cross Validation 5-fold + Holdout', fontsize=13)

# Plot CV por fold
ax = axes[0]
bars = ax.bar(range(1,6), fold_scores, color=COLORS[0], edgecolor='white', linewidth=0.8)
ax.axhline(np.mean(fold_scores), color='red', linestyle='--', lw=1.5,
           label=f'Media: {np.mean(fold_scores):.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy por fold (5-fold CV)')
ax.set_ylim(0, 1.1); ax.legend()
for bar, v in zip(bars, fold_scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{v:.3f}', ha='center', fontsize=10)

# Plot holdout
ax = axes[1]
metrs = ['ROUGE-1','ROUGE-L','BLEU','Recall@1','Recall@3']
x_pos = np.arange(len(metrs))
w = 0.25
for i, (label, m) in enumerate([('Baseline',m_base),('Var1',m_v1),('Var2',m_v2)]):
    vals = [m[mt] for mt in metrs]
    ax.bar(x_pos + i*w, vals, w, label=label, color=COLORS[i], edgecolor='white')
ax.set_xlabel('Métrica'); ax.set_ylabel('Score')
ax.set_title('Holdout test — métricas por variante')
ax.set_xticks(x_pos+w); ax.set_xticklabels(metrs, rotation=20)
ax.set_ylim(0,1.15); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../logs/validacion_cv_holdout_semana5.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Gráfica guardada: logs/validacion_cv_holdout_semana5.png')

---
## 8. Guardar log completo del experimento

In [ ]:
import json
from datetime import datetime

full_log = {
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'semana': 5,
    'split': {
        'train': len(X_train), 'val': len(X_val), 'test': len(X_test),
        'tipo': 'holdout aleatorio',
        'leakage': 'CERO — fit solo en train',
    },
    'experimentos': {
        'baseline': {
            'descripcion': 'TF-IDF unigrams sin normalización',
            'features_añadidas': [],
            'features_removidas': [],
            'metricas': m_base,
        },
        'variante1': {
            'descripcion': 'TF-IDF bigrams + strip_accents + sublinear_tf',
            'features_añadidas': ['bigrams(1,2)', 'strip_accents=unicode', 'sublinear_tf=True', 'max_features=8000'],
            'features_removidas': [],
            'metricas': m_v1,
        },
        'variante2': {
            'descripcion': 'TF-IDF bigrams + reranking por longitud + filtro similitud',
            'features_añadidas': ['reranking_longitud_doc', 'filtro_similitud>=0.05', 'max_features=10000'],
            'features_removidas': [],
            'metricas': m_v2,
        },
    },
    'cross_validation': {
        'tipo': 'StratifiedKFold 5-fold',
        'modelo': 'Naive Bayes (clasificación de intención)',
        'metricas': cv_metrics,
        'fold_accuracies': [round(s,4) for s in fold_scores],
    },
}

log_path = '../logs/experimentos_semana5.json'
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(full_log, f, indent=2, ensure_ascii=False)

print('✅ Log completo guardado en:', log_path)
print('\nResumen:')
print(f'  Baseline  → ROUGE-1={m_base["ROUGE-1"]} | Recall@3={m_base["Recall@3"]} | {m_base["Latencia_ms"]}ms')
print(f'  Variante1 → ROUGE-1={m_v1["ROUGE-1"]} | Recall@3={m_v1["Recall@3"]} | {m_v1["Latencia_ms"]}ms')
print(f'  Variante2 → ROUGE-1={m_v2["ROUGE-1"]} | Recall@3={m_v2["Recall@3"]} | {m_v2["Latencia_ms"]}ms')
print(f'  CV F1     → {cv_metrics["f1"]} (5-fold estratificado)')